# Market Basket Analysis

Hacemos conexión a SQL para extraer los datos

In [58]:
import sqlite3
import pandas as pd

In [59]:
"""
Aquí subí al colab la base de datos con extensión .db;
En cada sección hay que subir la base de datos y darle en el segundo ícono,
el que corresponde a una flecha curva
"""

conexion=sqlite3.connect('sanoyfresco.db')

In [60]:
DF=pd.read_sql_query('select * from tickets', conexion)
df=DF.copy()

In [61]:
#Se cierra la conexión
conexion.close()

Comenzamos con el análisis

In [62]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4975718 entries, 0 to 4975717
Data columns (total 11 columns):
 #   Column           Dtype  
---  ------           -----  
 0   id_pedido        int64  
 1   id_cliente       int64  
 2   fecha            object 
 3   hora             int64  
 4   id_departamento  int64  
 5   id_seccion       int64  
 6   id_producto      int64  
 7   nombre_producto  object 
 8   precio_unitario  float64
 9   cantidad         int64  
 10  precio_total     float64
dtypes: float64(2), int64(7), object(2)
memory usage: 417.6+ MB


In [63]:
#1. Cambiamos el tipo de dato de fecha: de object a datatime
df['fecha']=pd.to_datetime(df['fecha'])

In [64]:
"""
2. Extraemos la información que nos concierne para el análisis de la canasta de compra

Se hacen cálculos sobre la probabilidad de comprar un artículo por compra, y la
probabilidad conjunta de comprar dos productos por compras. Entonces las variables
a utilizar son:

*Pedido
*Nombre del producto

El objetivo de esto es optimizar los tiempos de cómputo.
"""


df_basket=df[['id_pedido','nombre_producto']]
df_basket

,id_pedido,nombre_producto
0,1,Pepino Kirby
1,1,Bolsa de Bananas Orgánicas
2,1,Aguacate Hass Orgánico
3,2,Col Rizada Orgánica de Michigan
4,2,Zanahorias
...,...,...
4975713,3421080,Leche Entera Orgánica
4975714,3421080,Racimo de Tomates Orgánicos
4975715,3421080,Cilantro Orgánico
4975716,3421082,Fresas


In [65]:
"""
2.1 Vamos a agrupar por pedido
"""

"""
Nota: para entender el algoritmo apropiado para hacer eso, repare en el siguiente
ejemplo. En el cuál se cuentan cuántos productos diferentes se compraron en el
pedido correspondiente:

"""
#df_basket.groupby('id_pedido')['nombre_producto'].count()


'''
Entonces, en la siguiente línea lo que se hace es: Agrupar por pedido, y en el
campo 'nombre producto' unir todos los productos correspondientes a dicho pedido
'''

df_pedido_agrupado=df_basket.groupby('id_pedido')['nombre_producto'].apply(lambda x: ','.join(x))
df_pedido_agrupado


,nombre_producto
id_pedido,
1,"Pepino Kirby,Bolsa de Bananas Orgánicas,Aguaca..."
2,"Col Rizada Orgánica de Michigan,Zanahorias"
3,Espinacas Baby Orgánicas
5,"Bolsa de Bananas Orgánicas,Frambuesas Orgánica..."
10,"Banana,Cilantro Orgánico,Aguacate Orgánico,Ceb..."
...,...
3421077,"Frambuesas Orgánicas,Calabacín Orgánico,Leche ..."
3421078,"Manzanas Gala Orgánicas,Banana"
3421080,"Leche Entera Orgánica,Racimo de Tomates Orgáni..."


In [66]:
'''
3.Quiero hacer una tabla de pedidos y productos.

Quiero que se marque por pedido, si un producto está presente o no
'''

'\n3.Quiero hacer una tabla de pedidos y productos.\n\nQuiero que se marque por pedido, si un producto está presente o no\n'

In [67]:
df_tabla_pedidoxproducto=df_pedido_agrupado.str.get_dummies(sep=',')
df_tabla_pedidoxproducto

,Agua con Gas de Pomelo,Aguacate Hass Orgánico,Aguacate Orgánico,Ajo Orgánico,Apio Orgánico en Ramillete Pequeño,Arándanos Orgánicos,Banana,Bolsa de Bananas Orgánicas,Calabacín Orgánico,Cebolla Amarilla Orgánica,...,Manzana Honeycrisp Orgánica,Manzanas Gala Orgánicas,Pepino Kirby,Pepino Orgánico,Racimo de Tomates Orgánicos,Rúcula Baby Orgánica,Tomates Cherry Orgánicos,Uvas Rojas sin Semillas,Zanahorias,Zanahorias Baby Orgánicas
id_pedido,,,,,,,,,,,,,,,,,,,,,
1,0,1,0,0,0,0,0,1,0,0,...,0,0,1,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,1,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
10,0,0,1,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3421077,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3421078,0,0,0,0,0,0,1,0,0,0,...,0,1,0,0,0,0,0,0,0,0
3421080,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0


## Se implementa la metodología del Market Basket Analysis

1. Creación del primero indicador: *el soporte*

In [68]:
'''
¿Cuál es la probabilidad individual de que se compre cada uno de los productos:

Camino 1:
1. Sumar todos los pedidos por cada producto.
Por ejemplo: si en el pedido 1 se pidió 1 Aguacate, y en el pedido 3 también,
Aguacate= 2 cantidades
'''

#1. Sumo los pedidos de cada producto
historico=df_tabla_pedidoxproducto.sum().rename('Histórico de veces comprado')
#historico

#1.5. De ese resultado surge una serie, la cuál la convierto a un dataframe de pandas
historicoprueba=historico.copy()
historicoprueba = pd.DataFrame(historicoprueba)

#2. A ese dataframe la añado una columna, que equivale a la probabilidad en % de compra
# de cada producto por pedido. NOTA. El índice de ese datafrase es el nombre del producto.
historicoprueba['Probabilidad']=(historicoprueba['Histórico de veces comprado']/len(df_tabla_pedidoxproducto))*100
historicoprueba

,Histórico de veces comprado,Probabilidad
Agua con Gas de Pomelo,79101,3.839504
Aguacate Hass Orgánico,220502,10.703004
Aguacate Orgánico,183961,8.929331
Ajo Orgánico,113759,5.521778
Apio Orgánico en Ramillete Pequeño,70621,3.427891
Arándanos Orgánicos,104784,5.086138
Banana,490518,23.809381
Bolsa de Bananas Orgánicas,394108,19.129710
Calabacín Orgánico,109240,5.302429
Cebolla Amarilla Orgánica,117561,5.706324


In [69]:
'''
Camino 2.

Una variable aleatoria es Bernulli si: x=1, si ocurre el evento;
                                       x=0, si no ocurre el evento.

La probabilidad de que ocurra el evento es: P(x=1)=p
La probabilidad de que NO ocurra el evento es: P(x=1)=1-p

El promedio E[x]=1*p+0(1-p)=p

La media coincide con la probabilidad de ocurrencia únicamente cuando la variable es binaria (0–1).


Así, el camino 2 consiste en hallar el promedio de cada producto.
'''

soporte=df_tabla_pedidoxproducto.mean()*100
soporte.sort_values(ascending=False)



,0
Banana,23.809381
Bolsa de Bananas Orgánicas,19.129710
Fresas Orgánicas,13.349170
Espinacas Baby Orgánicas,12.198935
Aguacate Hass Orgánico,10.703004
Aguacate Orgánico,8.929331
Limón Grande,7.791279
Fresas,7.241960
Limones,7.107944
Leche Entera Orgánica,6.918301


Entonces, el producto que más se compra son las bananas con un 23.8% de los pedidos; Luego las Bolsas de Bananas Orgánicas se compran en un 19.1% de los pedidos...

2. Creación del segundo indicador: Indicador de confianza

P(B|A)=P(A∩B)/P(A)

In [70]:
'''
1. Quiero crear el indicador de confianza.

Este calcula la probabilidad conjunta de que en una compra, se compren los dos
productos al tiempo.

Así que se deben hacer filtros como

df_tabla_pedidoxproducto[(df_tabla_pedidoxproducto['Agua con Gas de Pomelo']==1) &
 (df_tabla_pedidoxproducto['Aguacate Hass Orgánico']==1) ]

para encontrar cuáles mezclas de productos tienen mayor soporte conjunto.


Como se quiere hacer la convinación de ello, se procede a realizar una función:
'''

def indicador_confianza (antecedente, consecuente):
  veces_conjunta=df_tabla_pedidoxproducto[(df_tabla_pedidoxproducto[antecedente]==1) &
                           (df_tabla_pedidoxproducto[consecuente]==1) ]

  return len(veces_conjunta)/df_tabla_pedidoxproducto[antecedente].sum()


3. Creación del tercer indicador: indicador del fit

P(A∩B)/[P(A)⋅P(B)]

In [71]:
def indicador_fit(antecedente, consecuente):
  veces_conjunta=df_tabla_pedidoxproducto[(df_tabla_pedidoxproducto[antecedente]==1) &
                           (df_tabla_pedidoxproducto[consecuente]==1) ]
  probabilidad_conjunta=len(veces_conjunta)/len(df_tabla_pedidoxproducto)
  probabilidad_antecedente=df_tabla_pedidoxproducto[antecedente].mean()
  probabilidad_consecuente=df_tabla_pedidoxproducto[consecuente].mean()
  return probabilidad_conjunta/(probabilidad_antecedente*probabilidad_consecuente)



4. Agregar los indicadores a cada par de elementos

In [72]:
from itertools import combinations

#Vamos a decir que el umbral mínimo de confianza sea de 5%
umbral_confianza=0.05

asociaciones=[]

#Generamos las combinaciones de productos y se calcula la confianza y los lift
"""
De la tabla de dummies, se va a trabajar por columnas, exactamente de a dos columnas,
una de ellas va a ser 'antecedente' y otra 'consecuente'
"""

for antecedente, consecuente in combinations(df_tabla_pedidoxproducto.columns,2):

  #Se calcula el soporte del antecedente
  soporte_antecedente=df_tabla_pedidoxproducto[antecedente].mean()

  #Se calcula la confianza
  confianza=indicador_confianza(antecedente, consecuente)

  #Se hace un filtro para comprar la confianza calculada con el umbral de confianza
  if confianza>umbral_confianza:
  #si se cumple el filtro, se rellena la tabla asociaciones con la siguiente información
    asociaciones.append({
        'antecedente':antecedente,
        'consecuente':consecuente,
        'soporte_antecedente':round(soporte_antecedente*100,1),
        'confianza':round(confianza*100,1),
        'lift':round(indicador_fit(antecedente,consecuente),1)
    })



  df_asociaciones=pd.DataFrame(asociaciones)
  df_asociaciones.sort_values(by='lift', ascending=False, inplace=True)


In [73]:
 df_asociaciones


,antecedente,consecuente,soporte_antecedente,confianza,lift
185,Cebolla Roja Orgánica,Cilantro Orgánico,3.4,12.8,3.6
224,Cilantro Orgánico,Limones,3.5,25.4,3.6
61,Ajo Orgánico,Cebolla Amarilla Orgánica,5.5,20.1,3.5
106,Apio Orgánico en Ramillete Pequeño,Zanahorias,3.4,12.0,3.3
102,Apio Orgánico en Ramillete Pequeño,Pepino Orgánico,3.4,12.7,3.1
...,...,...,...,...,...
136,Bolsa de Bananas Orgánicas,Limón Grande,19.1,5.2,0.7
38,Aguacate Orgánico,Bolsa de Bananas Orgánicas,8.9,13.0,0.7
58,Ajo Orgánico,Banana,5.5,17.4,0.7
3,Agua con Gas de Pomelo,Bolsa de Bananas Orgánicas,3.8,13.2,0.7


### ¿Cómo se interpretan estos resultados?

Cuando el producto antecedente sea el Cilantro Orgánico (el cliente muestre interés en ese producto, lo añada al carrito, por ejemplo), se puede recomendar
que añada limones.

¿Por qué puedo decir eso?
Es que el cilantro orgánico se añade en un 3.5% de las compras que los clientes hacen en nuestro mercado (en los datos que tenemos). Y que además, cuando el cliente ha añadido cilantro orgánico, la probabilidad de que también esté interesado en comprar Limones es del 25.4%. Esa probabilidad del 25.4% es 3.6 más grande que si los añadiera de manera aleatoria.

5. Crear tabla que tenga los resultados obtenidos y unir los id de producto, secciones y departamento

In [74]:
productos=df[['id_producto','id_seccion','id_departamento', 'nombre_producto']].drop_duplicates()
productos

,id_producto,id_seccion,id_departamento,nombre_producto
0,49683,83,4,Pepino Kirby
1,13176,24,4,Bolsa de Bananas Orgánicas
2,47209,24,4,Aguacate Hass Orgánico
3,28985,83,4,Col Rizada Orgánica de Michigan
4,17794,83,4,Zanahorias
5,21903,123,4,Espinacas Baby Orgánicas
7,27966,123,4,Frambuesas Orgánicas
9,24852,24,4,Banana
10,31717,16,4,Cilantro Orgánico
11,47766,24,4,Aguacate Orgánico


Aquí voy a unir las tablas productos (con información útil) y el resultado que encontramos. Esos resultados le llamamos reglas de negocio.

In [75]:
df_asociaciones_enriquecido = df_asociaciones.merge(productos, left_on='antecedente', right_on='nombre_producto', how='left').drop(columns=['nombre_producto'])
df_asociaciones_enriquecido.columns = ['antecedente', 'consecuente', 'soporte_a', 'confianza', 'lift', 'id_producto_a', 'id_seccion_a', 'id_departamento_a']
df_asociaciones_enriquecido

,antecedente,consecuente,soporte_a,confianza,lift,id_producto_a,id_seccion_a,id_departamento_a
0,Cebolla Roja Orgánica,Cilantro Orgánico,3.4,12.8,3.6,8518,83,4
1,Cilantro Orgánico,Limones,3.5,25.4,3.6,31717,16,4
2,Ajo Orgánico,Cebolla Amarilla Orgánica,5.5,20.1,3.5,24964,83,4
3,Apio Orgánico en Ramillete Pequeño,Zanahorias,3.4,12.0,3.3,44359,83,4
4,Apio Orgánico en Ramillete Pequeño,Pepino Orgánico,3.4,12.7,3.1,44359,83,4
...,...,...,...,...,...,...,...,...
394,Bolsa de Bananas Orgánicas,Limón Grande,19.1,5.2,0.7,13176,24,4
395,Aguacate Orgánico,Bolsa de Bananas Orgánicas,8.9,13.0,0.7,47766,24,4
396,Ajo Orgánico,Banana,5.5,17.4,0.7,24964,83,4
397,Agua con Gas de Pomelo,Bolsa de Bananas Orgánicas,3.8,13.2,0.7,44632,115,7


Finalmente, descargamos las reglas de negocio, con información de interés.

In [76]:
df_asociaciones_enriquecido.to_csv('reglas.csv', index=False, sep=';', decimal=',')